In [5]:
from collections import defaultdict
import csv
from itertools import combinations 
import numpy as np

Transactions_list = []  # a list of transactions
Items_names = {}  # Lookup item ID to name
Items_ids = {}  # Lookup item name to ID

Items = None  # a list of item IDs, normally an increasing sequence of numbers

# Process the data
with open('./store_data.csv', 'r') as fin:
    reader = csv.reader(fin, delimiter=',')
    item_id = 0
    for row in reader:
        transaction = []
        for item in row:
            if item not in Items_ids:
                Items_ids[item] = item_id
                Items_names[item_id] = item
                item_id += 1
            transaction += [Items_ids[item]]
        Transactions_list += [transaction]

M, N = len(Items_ids), len(Transactions_list)

Items = np.arange(0,M)

# Information, sanity
print(f'M={M} items, N={N} transactions')

M=120 items, N=7501 transactions


In [6]:
# Sanity check
print([Items_names[_] for _ in Items[0:7]])
print(Transactions_list[:7])

['shrimp', 'almonds', 'avocado', 'vegetables mix', 'green grapes', 'whole weat flour', 'yams']
[[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19], [20, 21, 22], [23], [24, 2], [14, 25, 26, 27, 11], [10], [28, 29]]


In [7]:
# Convert to numpy arrays
Transactions = np.full((N,M), False, dtype=bool)

for i, t in enumerate(Transactions_list):
    for item in t:
        Transactions[i][item] = True

# Sanity, print row index 10, 11
print(f'{Transactions[10:12].astype(int)}')



[[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0]]


In [8]:
# Based on equations
def support(_itemset:list)->int:
    global Transactions, N, M
    cnt = 0
    for transaction_i in range(N):  # all transactions
        bMatch = True
        for item_ in _itemset:
            if not Transactions[transaction_i][item_]:
                bMatch = False
                break
        if bMatch:
            cnt += 1
    return cnt

def confidence(_it1:list, _it2:list)->float:
    return support([_it1, _it2])/support([_it1])

In [ ]:
Itemset_1, Itemset_2, Itemset_3 = {}, {}, {}


In [ ]:
%%time

# 1-itemset supports
items_1 = list(combinations(Items, 1))
print(f'Size of possible 1-items= {len(items_1)}')

for itemset in items_1:
    Itemset_1[itemset] = support(itemset)

# Convert back to item names and sort
sorted_1 = list(map(lambda x: (tuple(Items_names[_] for _ in x[0]),x[1]), Itemset_1.items()))
sorted_1 = sorted(sorted_1, key=lambda kv: kv[1], reverse=True)

# Print top-2 and bottom-2
print('\n'.join(map(str, sorted_1[:3]+sorted_1[-2:])))

In [ ]:
%%time

# 2-itemset supports
items_2 = list(combinations(Items, 2))
print(f'Size of possible 2-items= {len(items_2)}')

for itemset in items_2:
    Itemset_2[itemset] = support(itemset)

# Convert back to item names and sort
sorted_2 = list(map(lambda x: (tuple(Items_names[_] for _ in x[0]),x[1]), Itemset_2.items()))
sorted_2 = sorted(sorted_2, key=lambda kv: kv[1], reverse=True)

In [ ]:
# Find the first 0 support to mark the bottom
ix = next((i for i, x in enumerate([cnt for it,cnt in sorted_2]) if x==0), None)

# Print top-5 and bottom-2
print('\n'.join(map(str, sorted_2[:5]+sorted_2[ix-2:ix+1])))

In [ ]:
%%time

# 3-itemset supports
items_3 = list(combinations(Items, 3))
print(f'Size of possible 3-items= {len(items_3)}')

for itemset in items_3:
    Itemset_3[itemset] = support(itemset)

# Convert back to item names and sort
sorted_3 = list(map(lambda x: (tuple(Items_names[_] for _ in x[0]),x[1]), Itemset_3.items()))
sorted_3 = sorted(sorted_3, key=lambda kv: kv[1], reverse=True)

In [ ]:
# Find the first 0 support to mark the bottom
ix = next((i for i, x in enumerate([cnt for it,cnt in sorted_3]) if x==0), None)

# Print top-5 and bottom-2
print('\n'.join(map(str, sorted_3[:5]+sorted_3[ix-2:ix+1])))

In [ ]:
# 4-itemset supports
items_4 = list(combinations(Items, 4))
print(f'Size of possible 4-items= {len(items_4)}')

In [ ]:
# Find high confidence item in 2-itemsets
Conf_2 = defaultdict(list)

for itemX in Items:
    for itemY in Itemset_2:
        if itemX in itemY:
            Conf_2[itemX] += [(itemY, Itemset_2[itemY]/Itemset_1[(itemX,)])]

In [ ]:
# Sort each item's confidence to find the best for each item
for item in Items:
    Conf_2[item] = sorted(Conf_2[item], key=lambda kv: kv[1], reverse=True)
    
conf_top = np.array([Conf_2[i][0][1] for i in range(M)])

# Top confidence items for 2-itemsets
conf_top_top = conf_top.argsort()[-10:][::-1]
for ix in conf_top_top:
    print(f'{Conf_2[ix][0][1]:.3f} {Items_names[ix]:>15} -> {tuple(map(lambda x: Items_names[x], Conf_2[ix][0][0]))}')

In [ ]:
# Find high confidence item in 3-itemsets
Conf_3 = defaultdict(list)

for itemX in Items:
    for itemY in Itemset_3:
        if itemX in itemY:
            Conf_3[itemX] += [(itemY, Itemset_3[itemY]/Itemset_1[(itemX,)])]

In [ ]:
# Sort each item's confidence to find the best for each item
for item in Items:
    Conf_3[item] = sorted(Conf_3[item], key=lambda kv: kv[1], reverse=True)
    
conf_top = np.array([Conf_3[i][0][1] for i in range(M)])

# Top confidence items for 3-itemsets
conf_top_top = conf_top.argsort()[-10:][::-1]
for ix in conf_top_top:
    print(f'{Conf_3[ix][0][1]:.3f} {Items_names[ix]:>15} -> {tuple(map(lambda x: Items_names[x], Conf_3[ix][0][0]))}')

In [9]:
def save_file(_fn):
    global Transactions, Items_names, N, M
    with open(_fn, 'w') as fout:
        writer = csv.writer(fout, delimiter=',', quoting=csv.QUOTE_ALL, quotechar="'", lineterminator='\n')
        writer.writerow([Items_names[i] for i in range(M)])
        for i in range(N):
            writer.writerow(list(map(lambda x: '' if x == False else 'True',  Transactions[i])))

save_file('input_for_weka.csv')